In [1]:
import json
import os
import  numpy as np
import pandas as pd
from keras_preprocessing.image import img_to_array

from pycocotools.coco import COCO
from PIL import Image

import matplotlib.pyplot as plt


os.environ['SM_FRAMEWORK']='tf.keras'
import segmentation_models as sm
from tensorflow.keras import Model
from tensorflow.keras.utils import Sequence,load_img
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten
import ctypes
import tensorflow as tf

Segmentation Models: using `tf.keras` framework.


In [2]:
ctypes.windll.kernel32.SetThreadExecutionState(0x80000002)


-2147483648

In [3]:
test_img="./arcade/stenosis/test/images/"
train_img="./arcade/stenosis/train/images/"
val_img="./arcade/stenosis/val/images/"

In [4]:
js_train="./arcade/stenosis/train/annotations/train.json"
js_val="./arcade/stenosis/val/annotations/val.json"
js_test="./arcade/stenosis/test/annotations/test.json"

with open (js_train,"r") as f:
    js_tra=json.load(f)

with open (js_val,"r") as b:
    js_v=json.load(b)

with open (js_test,"r") as c:
    js_te=json.load(c)

In [5]:
print(f" count train:{len(os.listdir(train_img))}")
print(f" count val:{len(os.listdir(val_img))}")
print(f" count test:{len(os.listdir(test_img))}")

 count train:1000
 count val:200
 count test:300


In [ ]:
img_filers=os.listdir(train_img)
x_train=[]
coco=COCO(js_train)
img_ids=coco.getImgIds()
for img_id in img_ids:
    img_p=coco.loadImgs(img_id)[0]
    img=os.path.join(train_img,img_p["file_name"])
    img=load_img(img,target_size=(512,512),color_mode="grayscale")
    img=img_to_array(img)
    x_train.append(img)
x_train=np.array(x_train)

img_filers_test=os.listdir(test_img)
x_test=[]
for j in img_filers_test:
    img_p2=os.path.join(test_img,j)
    img2=load_img(img_p2,target_size=(512,512),color_mode="grayscale")
    img2=img_to_array(img2)
    x_test.append(img2)
x_test=np.array(x_test)

img_filers_val=os.listdir(val_img)
val_a=[]
for k in img_filers_val:
    img_p3=os.path.join(val_img, k)
    img3=load_img(img_p3,target_size=(512,512),color_mode="grayscale")
    img3=img_to_array(img3)
    val_a.append(img3)
val_a=np.array(val_a)



loading annotations into memory...
Done (t=0.02s)
creating index...
index created!


In [ ]:
print("train:",x_train.shape)
print("test:",x_test.shape)
print("val:",val_a.shape)

In [ ]:
from pycocotools.coco import COCO
import numpy as np

def make_masks(json_path):
    coco=COCO(json_path)
    img_ids=coco.getImgIds()
    masks=[]
    for img_id in img_ids:

        anns=coco.loadAnns(coco.getAnnIds(imgIds=img_id))

        mask=np.zeros((512,512))
        for ann in anns:
            o=coco.annToMask(ann)
            mask=np.maximum(mask,o)

        masks.append(mask)

    masks=np.array(masks)
    masks=np.expand_dims(masks,-1)

    return masks

In [ ]:
y_train=make_masks(js_train)
y_val=make_masks(js_val)
y_test=make_masks(js_test)

In [ ]:
print("val mask:",y_val.shape)
print("test mask:",y_test.shape)
print("train mask:",y_train.shape)

In [ ]:
print(x_train.shape)
print(y_train.shape)

In [ ]:
plt.imshow(x_train[0].squeeze(), cmap="gray")
plt.show()

plt.imshow(y_train[0].squeeze(), cmap="gray")
plt.show()

In [ ]:
plt.figure(figsize=(8,8))

plt.imshow(x_train[0].squeeze(), cmap="gray")
plt.imshow(y_train[0].squeeze(), alpha=0.5)

plt.show()

In [ ]:
x_train=x_train/255.0
val_a=val_a/255.0
x_test=x_test/255.0

In [ ]:
model=sm.Unet('resnet34',input_shape=(512,512,1),activation="sigmoid",encoder_weights=None,classes=1)

In [ ]:
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])


In [ ]:
history=model.fit(x_train,y_train,validation_data=(val_a,y_val),epochs=100,batch_size=4)

In [ ]:
model.save("heart.keras")

In [ ]:
os.system("shutdown /s /t 60")

In [ ]:
print(js_tra.keys())

print(js_tra["annotations"][0])

In [ ]:
print(y_train[0].max())
print(np.sum(y_train[0]))

In [ ]:
print(tf.config.list_physical_devices("GPU"))

In [ ]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())